# Final Ensemble Experiment

This notebook is the final stage of the optimization workflow. Run the three model-specific notebooks first:

1. `models_training/random-forest/random_forest_optimization_test.ipynb`
2. `models_training/svm/svm_nested_cv_optimization.ipynb`
3. `models_training/gradient-boosting/gradient_boosting_nested_cv_optimization.ipynb`

Then this notebook loads their best hyperparameters and feature sets, tunes soft-voting weights on race-level CV inside the train/development period, refits the three optimized models, and evaluates the ensemble once on the untouched holdout season.


# 1. Imports

In [ ]:
import json
import pathlib
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, RobustScaler, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")


# 2. Load Ensemble Configuration and Model Results

In [ ]:
start_path = pathlib.Path(__file__).resolve().parent if "__file__" in dir() else pathlib.Path.cwd().resolve()
for candidate in [start_path, *start_path.parents]:
    config_candidate = candidate / "json-parameters" / "ensemble" / "ensemble_optimization_params.json"
    if config_candidate.exists():
        REPO_ROOT = candidate
        CONFIG_PATH = config_candidate
        break
else:
    raise FileNotFoundError("Could not locate json-parameters/ensemble/ensemble_optimization_params.json")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

model_results = {}
for model_name, artifact_cfg in config["artifacts"].items():
    results_path = REPO_ROOT / artifact_cfg["results_path"]
    if not results_path.exists():
        raise FileNotFoundError(
            f"Missing optimized results for {model_name}: {results_path}. "
            "Run the model-specific optimization notebook first."
        )
    with open(results_path, "r") as f:
        model_results[model_name] = json.load(f)

print("Config loaded from:", CONFIG_PATH)
for name, result in model_results.items():
    print(name, "holdout macro F1:", result.get("final_holdout_metrics", {}).get("macro_f1"))


# 3. Load Data and Create Holdout Split

In [ ]:
DATA_PATH = REPO_ROOT / "dataset" / "outputs" / "prediction.csv"
df = pd.read_csv(DATA_PATH)

sort_cols = config["data"].get("sort_by", ["year", "round"])
df = df.sort_values(sort_cols + ["raceId", "driverId"]).reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"])

target_col = config["data"]["target_column"]
le = LabelEncoder()
y = le.fit_transform(df[target_col])

holdout_cfg = config["data"].get("holdout", {})
test_years = sorted(int(year) for year in holdout_cfg.get("test_years", [2025]))
exclude_years = sorted(int(year) for year in holdout_cfg.get("exclude_years", []))

test_mask = df["year"].isin(test_years)
exclude_mask = df["year"].isin(exclude_years)
train_dev_mask = ~(test_mask | exclude_mask)

train_dev_idx = np.flatnonzero(train_dev_mask.to_numpy())
holdout_idx = np.flatnonzero(test_mask.to_numpy())

y_train_dev = y[train_dev_idx]
df_train_dev = df.iloc[train_dev_idx].reset_index(drop=True)
y_holdout = y[holdout_idx]
df_holdout = df.iloc[holdout_idx].reset_index(drop=True)

print("Classes:", dict(enumerate(le.classes_)))
print("Train/development rows:", len(df_train_dev))
print("Holdout rows:", len(df_holdout))
print("Holdout years:", test_years)
print("Excluded years:", exclude_years)


# 4. Feature Preparation and Estimator Builders

In [ ]:
def make_race_time_series_splits(df_part, n_splits):
    ordered_races = (
        df_part[["raceId", "date"]]
        .drop_duplicates()
        .sort_values(["date", "raceId"])
        .reset_index(drop=True)
    )
    race_ids = ordered_races["raceId"].to_numpy()
    race_cv = TimeSeriesSplit(n_splits=n_splits)

    for train_race_idx, test_race_idx in race_cv.split(race_ids):
        train_races = set(race_ids[train_race_idx])
        test_races = set(race_ids[test_race_idx])
        train_idx = np.flatnonzero(df_part["raceId"].isin(train_races).to_numpy())
        test_idx = np.flatnonzero(df_part["raceId"].isin(test_races).to_numpy())
        yield train_idx, test_idx


def fit_transform_features(df_train, df_eval, raw_features, categorical_features):
    raw_train = df_train[raw_features].copy()
    raw_eval = df_eval[raw_features].copy()
    numeric_features = [col for col in raw_features if col not in categorical_features]

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if categorical_features:
        train_cat = encoder.fit_transform(raw_train[categorical_features].astype(str))
        eval_cat = encoder.transform(raw_eval[categorical_features].astype(str))
        encoded_names = encoder.get_feature_names_out(categorical_features).tolist()
    else:
        train_cat = np.empty((len(raw_train), 0))
        eval_cat = np.empty((len(raw_eval), 0))
        encoded_names = []

    X_train = pd.concat(
        [raw_train[numeric_features].reset_index(drop=True), pd.DataFrame(train_cat, columns=encoded_names)],
        axis=1,
    )
    X_eval = pd.concat(
        [raw_eval[numeric_features].reset_index(drop=True), pd.DataFrame(eval_cat, columns=encoded_names)],
        axis=1,
    )
    return X_train.values, X_eval.values, encoder, X_train.columns.tolist()




class SentinelMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self, feature_names=None, sentinel_columns=None, sentinel_value=-1):
        self.feature_names = feature_names
        self.sentinel_columns = sentinel_columns
        self.sentinel_value = sentinel_value

    def fit(self, X, y=None):
        X_arr = np.asarray(X, dtype=float)
        self.feature_names_ = list(self.feature_names) if self.feature_names is not None else [
            f"feature_{idx}" for idx in range(X_arr.shape[1])
        ]
        self.sentinel_columns_ = list(self.sentinel_columns or [])
        self.sentinel_indices_ = [self.feature_names_.index(col) for col in self.sentinel_columns_]
        self.medians_ = {}
        for col, idx in zip(self.sentinel_columns_, self.sentinel_indices_):
            values = X_arr[:, idx]
            valid_values = values[values != self.sentinel_value]
            self.medians_[idx] = float(np.median(valid_values)) if len(valid_values) else 0.0
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float).copy()
        indicators = []
        for idx in self.sentinel_indices_:
            missing_mask = X_arr[:, idx] == self.sentinel_value
            indicators.append(missing_mask.astype(float))
            X_arr[missing_mask, idx] = self.medians_[idx]
        if indicators:
            return np.hstack([X_arr, np.column_stack(indicators)])
        return X_arr


def build_random_forest(params):
    return RandomForestClassifier(
        **params,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )


def encode_class_weight_spec(spec):
    if spec is None or spec == "balanced":
        return spec
    if not isinstance(spec, dict):
        raise TypeError(f"Unsupported class_weight option: {spec!r}")
    unknown = sorted(set(spec) - set(le.classes_))
    if unknown:
        raise ValueError(f"Unknown class names in svc__class_weight: {unknown}")
    encoded = {}
    for class_name in le.classes_:
        class_idx = int(le.transform([class_name])[0])
        encoded[class_idx] = float(spec.get(class_name, 1.0))
    return encoded


def build_svm(params, feature_names=None, sentinel_columns=None):
    params = params.copy()
    if "svc__class_weight" in params:
        params["svc__class_weight"] = encode_class_weight_spec(params["svc__class_weight"])
    scaler_name = params.get("scaler")
    if scaler_name == "RobustScaler":
        params["scaler"] = RobustScaler()
    elif scaler_name == "StandardScaler":
        params["scaler"] = StandardScaler()

    return Pipeline([
        ("sentinel", SentinelMedianImputer(
            feature_names=feature_names,
            sentinel_columns=sentinel_columns,
            sentinel_value=-1,
        )),
        ("scaler", StandardScaler()),
        ("svc", SVC(
            probability=True,
            random_state=42,
        )),
    ]).set_params(**params)


def build_gradient_boosting(params):
    return GradientBoostingClassifier(random_state=42).set_params(**params)


builders = {
    "random_forest": build_random_forest,
    "svm": build_svm,
    "gradient_boosting": build_gradient_boosting,
}

inner_n_splits = config["cv"]["inner_n_splits"]
outer_n_splits = config["cv"]["outer_n_splits"]
outer_splits = list(make_race_time_series_splits(df_train_dev, outer_n_splits))
weight_candidates = [tuple(weights) for weights in config["weight_candidates"]]

print("Weight candidates:", weight_candidates)


# 5. Tune Ensemble Weights with Race-Level CV

In [ ]:
def weighted_proba(probabilities_by_model, weights):
    total = np.zeros_like(next(iter(probabilities_by_model.values())))
    weight_sum = float(sum(weights))
    for weight, model_name in zip(weights, ["random_forest", "svm", "gradient_boosting"]):
        total += weight * probabilities_by_model[model_name]
    return total / weight_sum


weight_scores = {weights: [] for weights in weight_candidates}
outer_predictions = []

for fold_num, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    fold_train_df = df_train_dev.iloc[train_idx].reset_index(drop=True)
    fold_test_df = df_train_dev.iloc[test_idx].reset_index(drop=True)
    y_train = y_train_dev[train_idx]
    y_test = y_train_dev[test_idx]

    fold_probabilities = {}
    for model_name, result in model_results.items():
        raw_features = result["raw_features_used"]
        categorical_features = result.get("categorical_features_encoded", [])
        params = result["final_hyperparameters"]

        X_train, X_test, _, _ = fit_transform_features(
            fold_train_df,
            fold_test_df,
            raw_features,
            categorical_features,
        )
        if model_name == "svm":
            model = builders[model_name](
                params,
                raw_features,
                result.get("sentinel_impute_columns", []),
            )
        else:
            model = builders[model_name](params)
        model.fit(X_train, y_train)
        fold_probabilities[model_name] = model.predict_proba(X_test)

    best_fold_score = -np.inf
    best_fold_weights = None
    best_fold_pred = None
    for weights in weight_candidates:
        proba = weighted_proba(fold_probabilities, weights)
        pred = np.argmax(proba, axis=1)
        score = f1_score(y_test, pred, average="macro")
        weight_scores[weights].append(score)
        if score > best_fold_score:
            best_fold_score = score
            best_fold_weights = weights
            best_fold_pred = pred

    outer_predictions.append((y_test, best_fold_pred))
    print(f"Fold {fold_num}/{outer_n_splits} | best weights = {best_fold_weights} | macro F1 = {best_fold_score:.4f}")

weight_summary_df = pd.DataFrame([
    {
        "weights": weights,
        "mean_macro_f1": float(np.mean(scores)),
        "std_macro_f1": float(np.std(scores)),
    }
    for weights, scores in weight_scores.items()
]).sort_values("mean_macro_f1", ascending=False)

best_weights = tuple(weight_summary_df.iloc[0]["weights"])
display(weight_summary_df)
print("Best CV weights:", best_weights)


# 6. Final Holdout Evaluation

In [ ]:
final_models = {}
final_encoders = {}
final_feature_names = {}
final_probabilities = {}

for model_name, result in model_results.items():
    raw_features = result["raw_features_used"]
    categorical_features = result.get("categorical_features_encoded", [])
    params = result["final_hyperparameters"]

    X_train, X_holdout, encoder, encoded_names = fit_transform_features(
        df_train_dev,
        df_holdout,
        raw_features,
        categorical_features,
    )
    if model_name == "svm":
        model = builders[model_name](
            params,
            raw_features,
            result.get("sentinel_impute_columns", []),
        )
    else:
        model = builders[model_name](params)
    model.fit(X_train, y_train_dev)

    final_models[model_name] = model
    final_encoders[model_name] = encoder
    final_feature_names[model_name] = encoded_names
    final_probabilities[model_name] = model.predict_proba(X_holdout)

holdout_proba = weighted_proba(final_probabilities, best_weights)
holdout_pred = np.argmax(holdout_proba, axis=1)

label_order_names = ["podium", "points", "no_points"]
label_order_names = [name for name in label_order_names if name in le.classes_]
label_order = le.transform(label_order_names)

holdout_metrics = {
    "accuracy": accuracy_score(y_holdout, holdout_pred),
    "balanced_accuracy": balanced_accuracy_score(y_holdout, holdout_pred),
    "macro_precision": precision_score(y_holdout, holdout_pred, average="macro"),
    "macro_recall": recall_score(y_holdout, holdout_pred, average="macro"),
    "macro_f1": f1_score(y_holdout, holdout_pred, average="macro"),
    "weighted_precision": precision_score(y_holdout, holdout_pred, average="weighted"),
    "weighted_recall": recall_score(y_holdout, holdout_pred, average="weighted"),
    "weighted_f1": f1_score(y_holdout, holdout_pred, average="weighted"),
    "matthews_corrcoef": matthews_corrcoef(y_holdout, holdout_pred),
    "cohen_kappa": cohen_kappa_score(y_holdout, holdout_pred),
}

holdout_report = classification_report(
    y_holdout,
    holdout_pred,
    labels=label_order,
    target_names=label_order_names,
    digits=4,
    output_dict=True,
)
per_label_metrics_df = (
    pd.DataFrame(holdout_report)
    .T
    .loc[label_order_names, ["precision", "recall", "f1-score", "support"]]
    .rename(columns={"f1-score": "f1_score"})
)
per_label_metrics_df[["precision", "recall", "f1_score"]] = per_label_metrics_df[["precision", "recall", "f1_score"]].round(4)
per_label_metrics_df["support"] = per_label_metrics_df["support"].astype(int)

holdout_cm = confusion_matrix(y_holdout, holdout_pred, labels=label_order)
holdout_cm_df = pd.DataFrame(
    holdout_cm,
    index=[f"true_{name}" for name in label_order_names],
    columns=[f"pred_{name}" for name in label_order_names],
)

print("Best ensemble weights:", best_weights)
print("\nOverall holdout metrics:")
display(pd.DataFrame([(k, round(v, 4)) for k, v in holdout_metrics.items()], columns=["metric", "value"]))
print("\nPer-label holdout metrics:")
display(per_label_metrics_df)
print("\nHoldout confusion matrix:")
display(holdout_cm_df)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
disp = ConfusionMatrixDisplay(confusion_matrix=holdout_cm, display_labels=label_order_names)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Final Ensemble - Holdout Confusion Matrix")
plt.tight_layout()
plt.show()


# 7. Save Ensemble Results

In [ ]:
MODEL_DIR = REPO_ROOT / "models_training" / "ensemble"
MODEL_PATH = MODEL_DIR / "final_soft_voting_ensemble.joblib"
RESULTS_PATH = REPO_ROOT / "json-parameters" / "ensemble" / "ensemble_optimization_results.json"

ensemble_artifact = {
    "models": final_models,
    "encoders": final_encoders,
    "feature_names": final_feature_names,
    "label_encoder": le,
    "weights": best_weights,
    "model_order": ["random_forest", "svm", "gradient_boosting"],
}
joblib.dump(ensemble_artifact, MODEL_PATH)

output_record = {
    "model_order": ["random_forest", "svm", "gradient_boosting"],
    "best_weights": list(best_weights),
    "weight_cv_summary": weight_summary_df.to_dict(orient="records"),
    "holdout_split": {
        "train_development_years": sorted(int(year) for year in df_train_dev["year"].unique()),
        "holdout_test_years": test_years,
        "excluded_years": exclude_years,
        "train_development_rows": int(len(df_train_dev)),
        "holdout_test_rows": int(len(df_holdout)),
        "train_development_races": int(df_train_dev["raceId"].nunique()),
        "holdout_test_races": int(df_holdout["raceId"].nunique()),
    },
    "base_model_results": {
        name: {
            "features_used": result["features_used"],
            "raw_features_used": result["raw_features_used"],
            "final_hyperparameters": result["final_hyperparameters"],
            "holdout_metrics": result.get("final_holdout_metrics"),
        }
        for name, result in model_results.items()
    },
    "final_holdout_metrics": {key: round(float(value), 4) for key, value in holdout_metrics.items()},
    "final_holdout_per_label_metrics": per_label_metrics_df.to_dict(orient="index"),
    "final_holdout_classification_report": holdout_report,
    "final_holdout_confusion_matrix": {
        "labels": label_order_names,
        "matrix": holdout_cm.tolist(),
    },
    "model_path": str(MODEL_PATH),
}

with open(RESULTS_PATH, "w") as f:
    json.dump(output_record, f, indent=2, default=str)

print(f"Ensemble model saved to: {MODEL_PATH}")
print(f"Ensemble results saved to: {RESULTS_PATH}")
print(json.dumps(output_record["final_holdout_metrics"], indent=2))
